In [4]:
from langchain_community.tools import tool , BaseTool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from pydantic import BaseModel, Field
from typing import Type, List, Literal, Dict, TypedDict

from langchain.chat_models import init_chat_model


In [5]:
import dotenv
dotenv.load_dotenv()

True

# Simple Tool call and tool execution

In [6]:

# create the tool with base model and base tool
class InputStructure(BaseModel):
  a: int =Field(..., description="A number to do some calculation")
  b: int =Field(..., description="A number to do some calculation")


class toolClass(BaseTool):
  name : str = "Multiplication"
  description: str = "Multiply 2 nums"

  args_schema: type[BaseModel] = InputStructure

  def _run(self,a: int,b: int)->int:
    return a*b
  
multiplicationFunc=toolClass()
print(multiplicationFunc.name)
print(multiplicationFunc.description)



#make the llm
llm = init_chat_model(model="llama-3.3-70b-versatile",
                      model_provider='groq')
#bind the llm with tool
llm_with_tool = llm.bind_tools([multiplicationFunc])

#define the query
query="What is the product of 3, 2 ?"
#convert the query ro human message and append in a list
message_list = [HumanMessage(query)]

#get the ai messages and append in message_list
ai_message = llm_with_tool.invoke(query)
message_list.append(ai_message)

#get the tool_messages and append in message list
arg_to_pass_to_tool=ai_message.tool_calls[0]
tool_message=multiplicationFunc.invoke(arg_to_pass_to_tool)

message_list.append(tool_message)


#pass the message list to the ll+tool
result=llm_with_tool.invoke(message_list)
result.content


Multiplication
Multiply 2 nums


'The product of 3 and 2 is 6.'